# 🔔 Stock Alert Manager
### Manage all your NSE stock alerts from this notebook
---
**Run Cell 1 first (Setup) every time you open this notebook.**
Then run only the cell you need.

| Cell | What it does |
|------|--------------|
| Cell 1 | Setup — run this first always |
| Cell 2 | View all active alerts |
| Cell 3 | Add price / support / resistance alert |
| Cell 4 | Add trendline alert |
| Cell 5 | Remove one alert |
| Cell 6 | Remove all alerts for a stock |
| Cell 7 | Alert statistics |
| Cell 8 | Push all changes to GitHub |

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 1 — SETUP (Run this first every time)
# ════════════════════════════════════════════════════════

import os, sys, json, subprocess

# ── Clone repo if not already present ───────────────────
GITHUB_USERNAME = 'TusharQLab'
GITHUB_TOKEN    = ''   # ← paste your GitHub PAT token here
REPO_NAME       = 'stock-alert-engine'
ROOT            = f'/content/{REPO_NAME}'

if not os.path.exists(ROOT):
    print('📥 Cloning repo...')
    os.system(f'git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git {ROOT}')
    print('✅ Repo cloned')
else:
    print('📥 Pulling latest...')
    os.chdir(ROOT)
    os.system(f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git')
    os.system('git pull origin main')
    print('✅ Up to date')

# ── Install dependencies ─────────────────────────────────
os.system('pip install yfinance pandas pyarrow pyyaml -q')

# ── Load path helpers ────────────────────────────────────
sys.path.insert(0, ROOT)
os.chdir(ROOT)

ALERTS_PATH = f'{ROOT}/config/alerts.json'

def load_alerts():
    with open(ALERTS_PATH) as f:
        return json.load(f)

def save_alerts(alerts):
    with open(ALERTS_PATH, 'w') as f:
        json.dump(alerts, f, indent=2)

def push_to_github():
    os.chdir(ROOT)
    os.system(f'git remote set-url origin https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git')
    os.system('git config user.name  "alert-bot"')
    os.system('git config user.email "alert-bot@github.com"')
    os.system('git add config/alerts.json data/alert_history.parquet')
    os.system('git commit --amend --no-edit')
    result = subprocess.run(['git','push','--force','origin','main'], capture_output=True, text=True)
    if result.returncode == 0:
        print('✅ Pushed to GitHub successfully')
    else:
        print('⚠️  Push failed:', result.stderr)

def view_alerts():
    alerts = load_alerts()
    if not alerts:
        print('No active alerts.')
        return
    print()
    print('=' * 55)
    print('📋 ACTIVE ALERTS')
    print('=' * 55)
    total = 0
    for stock, alert_list in sorted(alerts.items()):
        print(f'  {stock}')
        for i, a in enumerate(alert_list):
            atype = a['type']
            level = a.get('level', '—')
            if atype in {'TrendlineBreak','TrendlineBreakout','TrendlineBreakdown'}:
                p1 = a.get('point1',{})
                p2 = a.get('point2',{})
                print(f'    [{i}] {atype:<20} {p1.get("date")} @ {p1.get("price")} → {p2.get("date")} @ {p2.get("price")}')
            else:
                print(f'    [{i}] {atype:<20} {level}')
            total += 1
        print()
    print(f'  Stocks : {len(alerts)} | Total alerts : {total}')
    print('=' * 55)

def add_alert(stock, alert_type, level):
    alerts = load_alerts()
    stock  = stock.upper().strip()
    if stock not in alerts:
        alerts[stock] = []
    for a in alerts[stock]:
        if a.get('type') == alert_type and a.get('level') == level:
            print(f'⚠️  Already exists: {stock} {alert_type} @ {level}')
            return
    alerts[stock].append({'type': alert_type, 'level': level})
    alerts[stock] = sorted(alerts[stock], key=lambda x: x.get('level', 0))
    save_alerts(alerts)
    print(f'✅ Added : {stock} | {alert_type} @ {level}')

def add_trendline_alert(stock, alert_type, date1, price1, date2, price2, volume_factor=1.5):
    alerts = load_alerts()
    stock  = stock.upper().strip()
    if stock not in alerts:
        alerts[stock] = []
    alerts[stock].append({
        'type'         : alert_type,
        'point1'       : {'date': date1, 'price': float(price1)},
        'point2'       : {'date': date2, 'price': float(price2)},
        'volume_factor': volume_factor
    })
    save_alerts(alerts)
    print(f'✅ Trendline added : {stock} | {alert_type}')
    print(f'   Point1 : {date1} @ {price1}')
    print(f'   Point2 : {date2} @ {price2}')
    print(f'   Volume factor : {volume_factor}x')

def remove_alert(stock, index):
    alerts = load_alerts()
    stock  = stock.upper().strip()
    if stock not in alerts:
        print(f'⚠️  {stock} not found')
        return
    if index >= len(alerts[stock]):
        print(f'⚠️  Index {index} out of range — run view_alerts() to check index numbers')
        return
    removed = alerts[stock].pop(index)
    if len(alerts[stock]) == 0:
        del alerts[stock]
        print(f'🗑️  Removed {stock} [{removed["type"]} @ {removed.get("level","trendline")}] — stock removed (no alerts left)')
    else:
        print(f'🗑️  Removed {stock} [{removed["type"]} @ {removed.get("level","trendline")}] — {len(alerts[stock])} alerts remaining')
    save_alerts(alerts)

def remove_stock(stock):
    alerts = load_alerts()
    stock  = stock.upper().strip()
    if stock not in alerts:
        print(f'⚠️  {stock} not found')
        return
    count = len(alerts[stock])
    del alerts[stock]
    save_alerts(alerts)
    print(f'🗑️  Removed all {count} alerts for {stock}')

def alert_stats():
    alerts = load_alerts()
    from collections import Counter
    type_counts = Counter()
    for alert_list in alerts.values():
        for a in alert_list:
            type_counts[a['type']] += 1
    print()
    print('=' * 40)
    print('📊 ALERT STATISTICS')
    print('=' * 40)
    print(f'  Total stocks : {len(alerts)}')
    print(f'  Total alerts : {sum(len(v) for v in alerts.values())}')
    print()
    print('  By type:')
    for atype, count in type_counts.most_common():
        print(f'    {atype:<22} : {count}')
    print('=' * 40)

print()
print('=' * 40)
print('✅ SETUP COMPLETE')
print('   All functions loaded and ready')
print('=' * 40)
view_alerts()

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 2 — VIEW ALL ACTIVE ALERTS
# ════════════════════════════════════════════════════════
# Just run this cell — no changes needed

view_alerts()

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 3 — ADD PRICE / SUPPORT / RESISTANCE ALERT
# ════════════════════════════════════════════════════════
#
# Alert Types you can use:
#   Price      — triggers when price crosses this level
#   Support    — triggers when price falls to this level
#   Resistance — triggers when price rises to this level
#   Breakout   — triggers when price breaks above this level
#   Breakdown  — triggers when price breaks below this level
#   VolumeSpike — triggers when volume > X times average (set level = multiplier eg 3.0)
#
# FORMAT:
#   add_alert("STOCK_NAME", "AlertType", price_level)
#
# ── EXAMPLES (edit and run) ──────────────────────────────

add_alert('RELIANCE',   'Support',    1200.00)   # Alert when RELIANCE falls to 1200
add_alert('TCS',        'Resistance', 4200.00)   # Alert when TCS rises to 4200
add_alert('INFY',       'Price',      1500.00)   # Alert when INFY crosses 1500
add_alert('SBIN',       'Breakout',   1050.00)   # Alert when SBIN breaks above 1050
add_alert('HDFCBANK',   'Breakdown',  1600.00)   # Alert when HDFCBANK breaks below 1600
add_alert('BAJFINANCE', 'VolumeSpike', 3.0)      # Alert when volume > 3x average

# ── View updated alerts ──────────────────────────────────
view_alerts()

# ── Push to GitHub ───────────────────────────────────────
push_to_github()

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 4 — ADD TRENDLINE ALERT
# ════════════════════════════════════════════════════════
#
# How to find trendline points:
#   1. Open chart of the stock
#   2. Draw your trendline connecting two price points
#   3. Note the DATE and PRICE at each point
#   4. Paste below
#
# Alert Types:
#   TrendlineBreak    — triggers on either breakout or breakdown
#   TrendlineBreakout — triggers only when price breaks ABOVE trendline
#   TrendlineBreakdown — triggers only when price breaks BELOW trendline
#
# volume_factor = minimum volume spike needed to confirm break
#   1.5 = volume must be 1.5x average (default)
#   2.0 = volume must be 2x average (stricter)
#
# FORMAT:
#   add_trendline_alert(
#       stock, alert_type,
#       date1 YYYY-MM-DD, price1,    ← first point on trendline
#       date2 YYYY-MM-DD, price2,    ← second point on trendline
#       volume_factor
#   )
#
# ── EXAMPLE 1 — Descending trendline (resistance) ────────
# Price making lower highs — alert when it breaks above
add_trendline_alert(
    'RELIANCE', 'TrendlineBreakout',
    '2026-03-01', 1350,    # first high on chart
    '2026-05-01', 1280,    # second lower high on chart
    volume_factor=1.5
)

# ── EXAMPLE 2 — Ascending trendline (support) ────────────
# Price making higher lows — alert when it breaks below
add_trendline_alert(
    'TCS', 'TrendlineBreakdown',
    '2026-02-01', 3600,    # first low on chart
    '2026-04-01', 3750,    # second higher low on chart
    volume_factor=1.5
)

# ── View updated alerts ──────────────────────────────────
view_alerts()

# ── Push to GitHub ───────────────────────────────────────
push_to_github()

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 5 — REMOVE ONE ALERT
# ════════════════════════════════════════════════════════
#
# STEP 1 — Run view_alerts() first to see index numbers
# STEP 2 — Note the [index] shown next to the alert
# STEP 3 — Run remove_alert with stock name and index
#
# Example output from view_alerts():
#   RELIANCE
#     [0] Support    1200        ← index 0
#     [1] Resistance 1350        ← index 1
#     [2] Price      1300        ← index 2
#
# To remove Resistance at 1350:
#   remove_alert('RELIANCE', 1)
#
# ── First view alerts to find index ─────────────────────
view_alerts()

# ── Then remove — edit stock and index ──────────────────
remove_alert('RELIANCE', 1)   # ← change stock name and index number

# ── Push to GitHub ───────────────────────────────────────
push_to_github()

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 6 — REMOVE ALL ALERTS FOR A STOCK
# ════════════════════════════════════════════════════════
#
# This removes the stock completely from monitoring
# Use when you no longer want to track a stock at all
#
# ── Edit stock name and run ──────────────────────────────
remove_stock('WIPRO')   # ← change to stock you want to remove

# ── Push to GitHub ───────────────────────────────────────
push_to_github()

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 7 — ALERT STATISTICS
# ════════════════════════════════════════════════════════
# Shows count of each alert type and total stocks
# No changes needed — just run

alert_stats()

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 8 — PUSH CHANGES TO GITHUB MANUALLY
# ════════════════════════════════════════════════════════
# Run this any time you want to sync changes to GitHub
# (push is also automatic after every add/remove cell)

push_to_github()

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 9 — VIEW ALERT HISTORY
# ════════════════════════════════════════════════════════
# Shows all alerts that have triggered in the past

import pandas as pd, os
history_path = f'{ROOT}/data/alert_history.parquet'

if os.path.exists(history_path):
    df = pd.read_parquet(history_path)
    if df.empty:
        print('No alert history yet.')
    else:
        print(f'Total triggered alerts : {len(df)}')
        print()
        print(df.to_string(index=False))
else:
    print('No history file found yet.')